In [1]:
# 另一种插值方法的尝试
import os
import numpy as np
import PIL
from PIL import Image
import torch

In [2]:
folder_path = r'E:\working_stack\WT10 0.5mm_72h'
image_set = []

file_list = sorted(f for f in os.listdir(folder_path) if f.lower().endswith('.png'))

for file_name in file_list:
    img = np.array(Image.open(os.path.join(folder_path, file_name)))
    image_set.append(img)

print(f'共导入 {len(image_set)} 张图片')
print(f'第一张图片尺寸: {image_set[0].shape}')

共导入 59 张图片
第一张图片尺寸: (3571, 5113, 3)


In [3]:
# 以第一张图片的尺寸为标准 (H, W)
target_h, target_w = image_set[0].shape[0], image_set[0].shape[1]
target_size = (target_w, target_h)  # PIL 的尺寸格式是 (W, H)

for i in range(len(image_set)):
    img_pil = Image.fromarray(image_set[i])
    img_resized = img_pil.resize(target_size, Image.LANCZOS)
    image_set[i] = np.array(img_resized)

print(f'统一后的尺寸: {image_set[0].shape}')
print(f'共 {len(image_set)} 张图片已缩放')

统一后的尺寸: (3571, 5113, 3)
共 59 张图片已缩放


In [4]:
# 方法：Cubic，知道临近四个点的RGB值，用三次函数拟合
rgb_arrays = []

for i in range(10):
    # reshape 成 (像素总数, 3)，每行为一个像素的 [R, G, B]
    rgb = image_set[i].reshape(-1, 3)
    rgb_arrays.append(rgb)
    print(f'第 {i+1}个图片: 形状 {rgb.shape}, 数据类型 {rgb.dtype}')

print(f'\n共创建 {len(rgb_arrays)} 个数组')
print(f'像素 (0,0) 的 RGB: {rgb_arrays[0][0]}')
print(f'像素 (0,1) 的 RGB: {rgb_arrays[0][1]}')
print(f'像素 (1,0) 的 RGB: {rgb_arrays[0][11451400]}') 

第 1个图片: 形状 (18258523, 3), 数据类型 uint8
第 2个图片: 形状 (18258523, 3), 数据类型 uint8
第 3个图片: 形状 (18258523, 3), 数据类型 uint8
第 4个图片: 形状 (18258523, 3), 数据类型 uint8
第 5个图片: 形状 (18258523, 3), 数据类型 uint8
第 6个图片: 形状 (18258523, 3), 数据类型 uint8
第 7个图片: 形状 (18258523, 3), 数据类型 uint8
第 8个图片: 形状 (18258523, 3), 数据类型 uint8
第 9个图片: 形状 (18258523, 3), 数据类型 uint8
第 10个图片: 形状 (18258523, 3), 数据类型 uint8

共创建 10 个数组
像素 (0,0) 的 RGB: [0 0 0]
像素 (0,1) 的 RGB: [0 0 0]
像素 (1,0) 的 RGB: [34  0  0]


In [ ]:
def fit_cubic(rgb_1, rgb_2, rgb_3, rgb_4):
    """
    三次多项式拟合：f(z) = a*z^3 + b*z^2 + c*z + d
    输入：同一像素位置在 z = 0, 1, 2, 3 四个位置的 RGB 值（每个为长度 3 的数组）
    输出：R, G, B 三个分量各自的三次多项式系数 (a, b, c, d)组成的矩阵
    """
    # 四个采样点的 z 坐标
    z = np.array([0.0, 1.0, 2.0, 3.0])

    # 范德蒙德矩阵：每行 [z^3, z^2, z, 1]
    A = np.vstack([z**3, z**2, z, np.ones_like(z)]).T  # 形状 (4, 4)

    # 堆叠四个位置的 RGB 值：形状 (4, 3)，行对应 z=0..3，列对应 R/G/B
    values = np.vstack([rgb_1, rgb_2, rgb_3, rgb_4]).astype(np.float32)

    # 对 R、G、B 每个分量分别解线性方程组 A @ coef = values[:, k]
    R = np.linalg.solve(A, values[:, 0])  # R 通道系数 (a, b, c, d)
    G = np.linalg.solve(A, values[:, 1])  # G 通道系数 (a, b, c, d)
    B = np.linalg.solve(A, values[:, 2])  # B 通道系数 (a, b, c, d)

    # 封装成矩阵
    RGB = np.vstack([R , G , B])
    return RGB


In [ ]:
def render_pixel(RGB, z_calc):
    # 幂向量 [z^3, z^2, z, 1]
    powers = np.array([z_calc**3, z_calc**2, z_calc, 1.0])
    # 每个通道的点积：f_k(z) = a*z^3 + b*z^2 + c*z + d
    rgb_value = RGB @ powers
    return rgb_value

# ---------------- 快速验证 ----------------
# 取一个非纯黑像素做拟合验证（跳过全黑像素）
# p = None
# for idx in range(len(rgb_arrays[0])):
#     if rgb_arrays[0][idx].max() > 0:
#         p = idx
#         break

# if p is not None:
#     print(f'验证像素索引: {p}')
#     coef = fit_cubic(rgb_arrays[0][p], rgb_arrays[1][p], rgb_arrays[2][p], rgb_arrays[3][p])
#     print('系数矩阵 (3, 4)：\n', coef)

#     # 还原 z = 0, 1, 2, 3 处的值，应与原始 RGB 值一致
#     for z in [0.0, 1.0, 2.0, 3.0]:
#         print(f'z = {z}: 还原值 {render_pixel(coef, z)}, 原始值 {rgb_arrays[int(z)][p]}')

#     # 中点 z = 0.5 处的插值结果
#     print('z = 0.5 处的插值 RGB:', render_pixel(coef, 0.5))
# else:
#     print('前 4 张图全是黑色像素，无法验证')


# 循环渲染整张图片，大小3571*5113*3
# 以第 1 张插入图 I_in_0 为例：用 I_0, I_1, I_2, I_3 拟合，求 z = 0.5 处的值
# （z 轴标定：z(I_0)=0, z(I_1)=-1, ...，这里取正向坐标 0,1,2,3，I_in_0 位于 z=0.5）
N_PIXELS = 3571 * 5113
inserted = np.zeros((N_PIXELS, 3), dtype=np.uint8)   # 存放插值结果

for i in range(N_PIXELS):
    # 取该像素在 4 张图上的 RGB 值
    rgb_1 = rgb_arrays[0][i]
    rgb_2 = rgb_arrays[1][i]
    rgb_3 = rgb_arrays[2][i]
    rgb_4 = rgb_arrays[3][i]

    # 拟合三次多项式并计算 z = 0.5 处的 RGB
    coef = fit_cubic(rgb_1, rgb_2, rgb_3, rgb_4)
    value = render_pixel(coef, 0.5)

    # 裁剪到 [0, 255] 并取整，防止多项式过冲
    inserted[i] = np.clip(np.round(value), 0, 255)

# 还原成图片形状 (H, W, 3)
img_in_0 = inserted.reshape(3571, 5113, 3)
print(f'渲染完成: 形状 {img_in_0.shape}, 数据类型 {img_in_0.dtype}')
print(f'像素 (0,0) 的插值 RGB: {img_in_0[0, 0]}')


渲染完成: 形状 (3571, 5113, 3), 数据类型 uint8
像素 (0,0) 的插值 RGB: [0 0 0]


In [ ]:
# 填充（向量化优化版）
# 填充规则：I_a与I_{a+1}之间插入I_in_a，再统一标定z轴得到z(I_0) = 0 , z(I_{1}) = -1 , z(I_{2}) = -2 , z(I_{3}) = -3 ，... , z(I_{9}) = -9
# 要插入的图片I_in_a的z轴为z(I_a)-0.5
#           用I_a , I_{a+1} , I_{a+2} , I_{a+3}的RGB值，调用fit_cubic方法拟合出I_in_a的RGB值(a<=6时)
#           或者用I_6, I_7,I_8,I_9的RGB值，调用fit_cubic方法拟合出I_in_a的RGB值(a=7 & 8)

import time

# ---- 向量化原理 ----
# 逐像素路径：coef = A^{-1} @ values（fit_cubic），value = powers^T @ coef（render_pixel）
# 两步合并：value = powers^T @ A^{-1} @ values = w^T @ values，其中 w = A^{-T} @ powers
# 关键：权重 w 只依赖 z_calc，与像素无关 → 整张图的插值 = 4 张源图的逐像素加权求和
z_nodes = np.array([0.0, 1.0, 2.0, 3.0])
A = np.vstack([z_nodes**3, z_nodes**2, z_nodes, np.ones_like(z_nodes)]).T  # (4, 4) 范德蒙德矩阵

def interp_image_fast(idx4, z_calc):
    """向量化插值：用 idx4 的 4 张源图拟合三次曲线，返回 z = z_calc 处的整张图 (H, W, 3)
    数学上等价于对每个像素分别调用 fit_cubic + render_pixel"""
    powers = np.array([z_calc**3, z_calc**2, z_calc, 1.0])
    w = np.linalg.solve(A.T, powers)            # Lagrange 权重 (4,)，整张图只需算一次
    out = np.zeros(rgb_arrays[0].shape, dtype=np.float32)
    for w_k, k in zip(w, idx4):
        out += rgb_arrays[k].astype(np.float32) * np.float32(w_k)
    return np.clip(np.round(out), 0, 255).astype(np.float32)

H, W = 3571, 5113
inserted_images = {}    # inserted_images[a] 即 I_in_a

t0 = time.time()
for a in range(9):
    if a <= 6:
        idx4 = [a, a + 1, a + 2, a + 3]    # I_a, I_{a+1}, I_{a+2}, I_{a+3}
        z_calc = 0.5                        # I_in_a 在拟合坐标下位于两图中点
    else:                                   # a = 7, 8：越界部分统一用 I_6..I_9
        idx4 = [6, 7, 8, 9]
        z_calc = (a - 6) + 0.5              # a=7 → 1.5, a=8 → 2.5
    inserted_images[a] = interp_image_fast(idx4, z_calc).reshape(H, W, 3)
    print(f'I_in_{a}: 源图 {idx4}, z_calc = {z_calc}, 累计用时 {time.time() - t0:.1f}s')

# ---- 正确性验证：与逐像素循环算出的 I_in_0 对比 ----
if 'img_in_0' in globals():
    diff = np.abs(inserted_images[0].astype(np.int16) - img_in_0.astype(np.int16))
    n_diff = int((diff > 0).sum())
    print(f'\n与逐像素循环的 I_in_0 对比: 不一致元素 {n_diff} 个 / {diff.size} 个, 最大差异 {int(diff.max())}')
else:
    print('\n(内核中没有 img_in_0，跳过对比——请先运行上一个单元格)')


I_in_0: 源图 [0, 1, 2, 3], z_calc = 0.5, 累计用时 1.2s
I_in_1: 源图 [1, 2, 3, 4], z_calc = 0.5, 累计用时 2.2s
I_in_2: 源图 [2, 3, 4, 5], z_calc = 0.5, 累计用时 3.2s
I_in_3: 源图 [3, 4, 5, 6], z_calc = 0.5, 累计用时 4.2s
I_in_4: 源图 [4, 5, 6, 7], z_calc = 0.5, 累计用时 5.3s
I_in_5: 源图 [5, 6, 7, 8], z_calc = 0.5, 累计用时 6.3s
I_in_6: 源图 [6, 7, 8, 9], z_calc = 0.5, 累计用时 7.4s
I_in_7: 源图 [6, 7, 8, 9], z_calc = 1.5, 累计用时 8.2s
I_in_8: 源图 [6, 7, 8, 9], z_calc = 2.5, 累计用时 9.1s

与逐像素循环的 I_in_0 对比: 不一致元素 264761 个 / 54775569 个, 最大差异 1


In [26]:
# 在此处下采样+渲染3D重构模型
# 参考 work1 的渲染方式：每张切片 = 一个带纹理的平面，离屏渲染导出静态 PNG
# 切片序列：I_0, I_in_0, I_1, I_in_1, ..., I_8, I_in_8, I_9（共 19 张）
# z 轴标定放大 4 倍：z(I_a) = -4a，z(I_in_a) = -4a - 2（切片间距 0.5 → 2，跨度 [-36, 0]）
import pyvista as pv

# ---------------- 参数 ----------------
RENDER_SCALE = 4                 # 渲染降采样倍数（减小显存占用；1 = 原始分辨率）
Z_DISPLAY_SCALE = 15             # 仅影响显示：z 拉伸倍数（标定放大后跨度 36 单位，仍需拉伸才够醒目）
SLICE_OPACITY = 0.25             # 切片不透明度（调小 = 更透明的"体积"效果）
OUTPUT_PNG = r'E:\SRT\stack3d_cubic.png'

# ---------------- 组装切片序列 ----------------
# z 轴标定（放大 4 倍）：z(I_a) = -4a，z(I_in_a) = -4a - 2（z 轴向下递减）
# 只改渲染用的 z 坐标；插值单元格的拟合坐标（z_calc = 0.5 等）保持原样
Z_CALIB_SCALE = 4
slices = []
for a in range(10):
    slices.append((-Z_CALIB_SCALE * a, image_set[a]))                     # 原始切片 I_a
    if a < 9:
        slices.append((-Z_CALIB_SCALE * (a + 0.5), inserted_images[a]))   # 插值切片 I_in_a

print(f'共 {len(slices)} 张切片（10 张原图 + 9 张三次插值）')
zs = [z for z, _ in slices]
print(f'z 范围: [{min(zs)}, {max(zs)}]，是否均匀间隔 {Z_CALIB_SCALE * 0.5}: {np.allclose(np.diff(sorted(zs)), Z_CALIB_SCALE * 0.5)}')

# ---------------- 渲染 ----------------
plotter = pv.Plotter(off_screen=True, window_size=(1920, 1440))
plotter.set_background('black')
# 注意：不要使用 enable_depth_peeling()！
# 在当前离屏渲染环境下深度剥离不可用，会导致输出全黑（work1 已实测验证）

n_added = 0
for z, img in slices:
    # 渲染降采样：大幅减小纹理显存占用（RENDER_SCALE=4 → 约 893×1278）
    h, w = img.shape[0] // RENDER_SCALE, img.shape[1] // RENDER_SCALE
    img_small = np.array(Image.fromarray(img).resize((w, h), Image.BILINEAR))

    # 平面位于显示坐标 z * Z_DISPLAY_SCALE；平面边长取像素数，保持像素纵横比 1:1
    plane = pv.Plane(center=(0, 0, z * Z_DISPLAY_SCALE),
                     direction=(0, 0, 1), i_size=w, j_size=h)
    plane.texture_map_to_plane(inplace=True)      # 生成 UV 纹理坐标

    # VTK 纹理 v=0 位于底部，而图像第 0 行在顶部 → 垂直翻转一次以保持正立
    tex = pv.numpy_to_texture(np.ascontiguousarray(img_small[::-1]))
    plotter.add_mesh(plane, texture=tex, opacity=SLICE_OPACITY)

    n_added += 1
    del img_small                     # 即用即弃，控制内存

print(f'已添加 {n_added} 张切片平面')
plotter.camera_position = [
    (-5500, -3800, 2000),    # 相机位置：负 x、负 y 象限，略高于堆叠
    (0, 0, -270),            # 焦点：堆叠几何中心（z 范围 [-36, 0] 的中点 -18 × 显示倍数 15）
    (0, 0, 1)                # 上方向
]
plotter.screenshot(OUTPUT_PNG)
print(f'渲染完成，已保存至 {OUTPUT_PNG}')


共 19 张切片（10 张原图 + 9 张三次插值）
z 范围: [-36, 0]，是否均匀间隔 2.0: True
已添加 19 张切片平面
渲染完成，已保存至 E:\SRT\stack3d_cubic.png
